In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02_transform_silver - Bronze → Silver
# MAGIC Estandarizar schemas y consolidar órdenes (Dataset 2016-2018)

# COMMAND ----------

from pyspark.sql import functions as F
from datetime import datetime

BRONZE_PATH = "/Volumes/olist/olist_bronze/bronze/"
SILVER_PATH = "/Volumes/olist/olist_silver/silver/"

start_time = datetime.now()
print(f"🚀 Inicio: {start_time.strftime('%H:%M:%S')}\n")

# COMMAND ----------

# Crear estructura Silver
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS olist.olist_silver")
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_silver.silver")
    print("✅ Volume Silver verificado\n")
except:
    pass

# COMMAND ----------

# Helper functions
def load(table):
    path = f"{BRONZE_PATH}{table}/"
    try:
        return spark.read.format("delta").load(path)
    except:
        return spark.read.parquet(path)

def save(df, table):
    spark.read.format("delta").load(f"{SILVER_PATH}{table}/")
    df.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}{table}/")

# COMMAND ----------

# Cargar Bronze
print("📥 Cargando Bronze...\n")

orders = load("orders")
order_items = load("order_items")
order_payments = load("order_payments")
order_reviews = load("order_reviews")
customers = load("customers")
products = load("products")
sellers = load("sellers")

print("✅ Cargado\n")

# COMMAND ----------

# Limpiar orders (fechas a timestamp)
print("🔄 Transformando orders...\n")

orders_clean = orders \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date")) \
    .filter(F.col("order_id").isNotNull())

print(f"✅ {orders_clean.count():,} orders\n")

# COMMAND ----------

# Agregaciones por order_id
print("📊 Creando agregaciones...\n")

# Items
items_agg = order_items \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double")) \
    .groupBy("order_id").agg(
        F.sum("price").alias("sum_price"),
        F.sum("freight_value").alias("sum_freight"),
        F.count("*").alias("items_count"),
        F.countDistinct("product_id").alias("distinct_products")
    )

# Payments
payments_agg = order_payments \
    .withColumn("payment_value", F.col("payment_value").cast("double")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("int")) \
    .groupBy("order_id").agg(
        F.sum("payment_value").alias("payment_sum"),
        F.avg("payment_installments").alias("avg_installments"),
        F.countDistinct("payment_type").alias("n_payment_types")
    )

# Reviews
reviews_agg = order_reviews \
    .withColumn("review_score", F.expr("try_cast(review_score as int)")) \
    .filter(F.col("review_score").isNotNull()) \
    .groupBy("order_id").agg(
        F.avg("review_score").alias("avg_review_score"),
        F.count("*").alias("review_count")
    )

print("✅ Agregaciones creadas\n")

# COMMAND ----------

# Crear orders_full
print("🔗 Creando orders_full...\n")

orders_full = orders_clean \
    .join(customers.dropDuplicates(["customer_id"]), "customer_id", "left") \
    .join(items_agg, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews_agg, "order_id", "left")

print(f"✅ {orders_full.count():,} registros, {len(orders_full.columns)} columnas\n")

# COMMAND ----------

# Control de calidad
print("🔍 Control de calidad:\n")

nulls = orders_full.filter(F.col("order_id").isNull()).count()
dups = orders_full.groupBy("order_id").count().filter(F.col("count") > 1).count()

print(f"{'✅' if nulls == 0 else '⚠️'} Nulls: {nulls:,}")
print(f"{'✅' if dups == 0 else '⚠️'} Duplicados: {dups:,}\n")

# COMMAND ----------

# Guardar en Silver
print("💾 Guardando...\n")

tables = {
    "orders": orders_clean,
    "orders_full": orders_full,
    "customers": customers.dropDuplicates(["customer_id"]),
    "products": products.dropDuplicates(["product_id"]),
    "sellers": sellers.dropDuplicates(["seller_id"])
}

for name, df in tables.items():
    save(df, name)
    print(f"✅ {name}: {df.count():,}")

# COMMAND ----------

# Resumen
duration = (datetime.now() - start_time).total_seconds()
print(f"\n⏱️  {duration:.2f} seg | ✅ {len(tables)} tablas en Silver")